# Inspect base activations

Load `onnx_models/smoll/base_activations.h5` (produced by
`scripts/generate_base_activations.py`) and look at a few samples: the base
token sequences and a couple of activation tensors at a couple of layers.

In [1]:
import h5py
import ml_dtypes
import numpy as np

H5_PATH = "onnx_models/smoll/base_activations.h5"

In [2]:
with h5py.File(H5_PATH, "r") as h5f:
    base_seqs = h5f["base_seqs"][:]
    activation_names = sorted(h5f["activations"].keys())

print("base_seqs:", base_seqs.shape, base_seqs.dtype)
print("activations available:", activation_names)

base_seqs: (1000, 8) int64
activations available: ['add_102', 'add_104', 'add_108', 'add_188', 'add_192', 'add_194', 'add_6', 'add_8', 'embedding']


## A few base sequences (token ids)

In [3]:
n_show = 5
base_seqs[:n_show]

array([[  446,   717,  1727,   105,  5484, 34471,  8140,   446],
       [ 6014,    90, 15037,   787,   717,  1727,   105,  5484],
       [  378,  3133,  2585,  1421,   281,   216,    34,    32],
       [  657,  1278,   351,  2731,  7122,   281,  3085,  3297],
       [ 1032,   351,  2672,   717,  1727,   105,  3792, 34471]])

## Load one activation tensor

Activations are stored as raw uint16 (h5py has no native bfloat16 type) --
view back to `ml_dtypes.bfloat16` on load, same as
`scripts/compute_base_jacobian_stats.py` does.

In [4]:
def load_activation(name: str) -> np.ndarray:
    with h5py.File(H5_PATH, "r") as h5f:
        return h5f["activations"][name][:].view(ml_dtypes.bfloat16)

layer_name = activation_names[0]
act = load_activation(layer_name)
print(layer_name, act.shape, act.dtype)

add_102 (1000, 8, 960) bfloat16


In [5]:
sample_idx = 0
seq_pos = 0

act[sample_idx, seq_pos, :16].astype(np.float32)

array([  2.6875    ,  -4.09375   ,  13.375     ,   0.08203125,
         5.84375   ,  13.3125    , -20.375     ,  22.875     ,
        18.5       ,   7.09375   ,   0.94921875, -22.375     ,
         5.84375   , -15.6875    ,  15.5       ,   5.3125    ],
      dtype=float32)

## Compare a few layers at the same sample/position

In [6]:
sample_idx = 0
seq_pos = 0

for name in activation_names:
    act = load_activation(name)
    vec = act[sample_idx, seq_pos].astype(np.float32)
    print(f"{name:>10s}  shape={act.shape}  mean={vec.mean():.4g}  std={vec.std():.4g}  norm={np.linalg.norm(vec):.4g}")

   add_102  shape=(1000, 8, 960)  mean=-16.78  std=772.1  norm=2.393e+04
   add_104  shape=(1000, 8, 960)  mean=-16.73  std=772  norm=2.393e+04


   add_108  shape=(1000, 8, 960)  mean=-16.75  std=772  norm=2.392e+04
   add_188  shape=(1000, 8, 960)  mean=-13.92  std=674.3  norm=2.09e+04
   add_192  shape=(1000, 8, 960)  mean=-13.57  std=666.2  norm=2.065e+04
   add_194  shape=(1000, 8, 960)  mean=-1.533  std=29.63  norm=919.3
     add_6  shape=(1000, 8, 960)  mean=-0.008571  std=0.2692  norm=8.344
     add_8  shape=(1000, 8, 960)  mean=-0.09303  std=3.864  norm=119.8
 embedding  shape=(1000, 8, 960)  mean=-0.004722  std=0.1463  norm=4.535
